In [ ]:
import matplotlib.pyplot as plt
import numpy as np

GAMMA, SLIP = 0.9, 0.1                                   # 折扣；打滑、留在原地的概率
GOAL, STEP = 10.0, -1.0                                  # 走进终点得 10 分，其余每一步扣 1 分


def p_right(theta):
    return 1 / (1 + np.exp(-theta))                      # 一个参数控制策略：往右的概率


p_right(1.4)                                             # 约 0.8

In [ ]:
rng = np.random.default_rng(0)


def episode(theta):
    s, ret, discount = 1, 0.0, 1.0                       # 下标 0、1、2 是 s1、s2、s3，从中间的 s2 出发
    while True:
        a = 1 if rng.random() < p_right(theta) else -1   # 往右 +1，往左 −1
        if rng.random() > SLIP:                          # 没打滑才移动；在 s1 往左撞墙，留在原地
            s = max(s + a, 0)
        if s == 3:                                       # 走进终点
            return ret + discount * GOAL
        ret += discount * STEP
        discount *= GAMMA


[round(episode(1.4), 2) for _ in range(5)]               # 同一个策略，每一局的回报都不同

In [ ]:
np.mean([episode(1.4) for _ in range(10000)])            # 跑一万局取平均：期望回报约 5.7

In [ ]:
def J(theta):
    V = np.zeros(3)                                      # s1、s2、s3 的价值
    for _ in range(200):                                 # 反复代入贝尔曼方程，直到不再变化
        new = np.zeros(3)
        for s in range(3):
            for a, p_a in ((1, p_right(theta)), (-1, 1 - p_right(theta))):
                moved = max(s + a, 0)
                for s_next, p in ((moved, 1 - SLIP), (s, SLIP)):
                    r, v_next = (GOAL, 0.0) if s_next == 3 else (STEP, V[s_next])
                    new[s] += p_a * p * (r + GAMMA * v_next)
        V = new
    return V[1]                                          # 从 s2 出发的期望回报


J(1.4), J(0.0)

In [ ]:
theta = np.linspace(-2, 4, 61)
plt.plot(theta, [J(t) for t in theta])                   # 期望回报是 θ 的函数
plt.xlabel("theta")
plt.ylabel("J")

In [ ]:
def grad(theta, h=1e-4):
    return (J(theta + h) - J(theta - h)) / (2 * h)       # 曲线在这一点的斜率


grad(1.4)                                                # 约 1.98：往大调，回报会涨

In [ ]:
theta, beta = 1.4, 0.5                                   # 起点与步长
for _ in range(4):
    theta += beta * grad(theta)                          # 沿着回报变大的方向走一步
    print(round(theta, 2), round(J(theta), 2))

In [ ]:
A, beta = -1.57, 10.0                                    # 打滑那一次的优势估计；很大的步长
theta_bad = 1.4 + beta * A * (1 - p_right(1.4))          # 往右这个动作推参数的方向：1 − 往右概率
theta_bad, p_right(theta_bad), J(theta_bad)              # 往右概率掉到 0.15 左右，期望回报跌到负的

In [ ]:
EPS = 0.2


def ppo_objective(ratio, adv):
    return np.minimum(ratio * adv, np.clip(ratio, 1 - EPS, 1 + EPS) * adv)   # 比值乘优势，与裁剪后的取较小者


ratio = np.linspace(0.4, 1.6, 121)
for adv in (1.04, -1.57):                                # 好样本、坏样本
    plt.plot(ratio, ppo_objective(ratio, adv), label=f"A = {adv}")
plt.xlabel("ratio")
plt.legend()

In [ ]:
ratio_bad = p_right(theta_bad) / p_right(1.4)            # 那一大步之后往右的概率比值
ratio_bad, ppo_objective(ratio_bad, A)                   # 远低于 0.8：目标已经变平，这个样本不再推动参数